IMPORTS

In [23]:
import pandas as pd
import ollama
import yaml

TESTING

In [15]:
test = pd.read_csv("./radiology-reporting-harness/test.csv")
test.head()

,case_id,modality,body_part,study_description,patient_age_band,patient_sex,template_content,dictation
0,00320399-b3ed-447c-be07-a3e3b9af6e2e,CT,Abdomen,CT A/P WO-B,60-64,female,FINDINGS:\nLIVER: Normal in size and attenuati...,Surgically absent gallbladder. Surgical clips ...
1,01b1c9ab-800b-4963-b11d-9d503952e57c,XRAY,Lumbar spine,XR LSP 4V,40-44,female,FINDINGS:\nVERTEBRAE: Normal desnity and align...,Mild lower lumbar facet degenerative changes.\...
2,02aae38a-088b-4cae-bfdc-f0fb38da565d,XRAY,Thoracic spine,XR TSP 2V,50-54,female,FINDINGS:\nVERTEBRAE: Normal desnity and align...,"degen chnges , mild scoliosis convexity to left"
3,045ea333-6d30-4edf-9d33-8ed3feb0772d,MRI,Lumbar spine,MRI LUMBAR,50-54,male,FINDINGS:\nVERTEBRAE: Normal vertebral body he...,Straightening of the normal lumbar lordosis is...
4,06ef946f-5b3b-4b7c-a64f-58c184dca4e6,XRAY,Shoulder,XR L SHOULDER 2V,65-69,male,FINDINGS:\nBONES: No fracture or focal lesion....,Bones show no acute fracture or dislocation. G...


In [10]:
test['template_content'][0].split("\n")

['FINDINGS:',
 'BONES: No acute fracture or focal osseous lesion.',
 'JOINTS: No dislocation. The joint spaces are normal.',
 'SOFT TISSUES: The soft tissues are unremarkable.',
 '',
 'IMPRESSION:',
 'No acute osseous abnormality.']

In [11]:
test['dictation'][0].split("\n")

['No acute fracture or dislocation. Mild right hip osteoarthrosis with mild superolateral joint space narrowing and small marginal acetabular and femoral head osteophytes. Mild degenerative changes of the bilateral sacroiliac joints Mild degenerative changes of the visualized lower lumbar spine. Pelvic ring is intact.']

In [12]:
train = pd.read_csv("./radiology-reporting-harness/train.csv")
train.head()

,case_id,modality,body_part,study_description,patient_age_band,patient_sex,template_content,dictation,report
0,2c88d015-b359-4e2c-9c62-61a23ce9d1c3,XRAY,Hip,XR RT HIP 2V,90+,female,FINDINGS:\nBONES: No acute fracture or focal o...,No acute fracture or dislocation. Mild right h...,FINDINGS:\nBONES: No acute fracture. Mild dege...
1,218fc4a5-c7e9-44a7-9aea-54e56bb22663,XRAY,Chest,XR CXR 2V,65-69,male,FINDINGS:\nLUNGS: Lungs are clear. No focal ai...,"no effusuon, infiltrates\nmild thoracic spondy...",FINDINGS:\nLUNGS: Lungs are clear. No focal ai...
2,2066ed76-9f41-4ac8-95f6-74ec1e4cef8a,MRI,Shoulder,MRI RT SHOULDER WO,45-49,male,FINDINGS:\nTENDONS:\nSUPRASPINATUS: The tendon...,MRI RIGHT SHOULDER WITHOUT CONTRAST Right shou...,FINDINGS:\nTENDONS:\nSUPRASPINATUS: There is m...
3,71210e04-aa16-4a47-8c20-fe58494260dc,MRI,Head,MRI Brain^SUBTLE BRAIN,70-74,female,FINDINGS:\nBRAIN: No restricted diffusion to i...,"mild atrophy , mild leuko",FINDINGS:\nBRAIN: There is mild atrophy. There...
4,656574f1-ffea-49f7-a0c8-953a5cd40bfc,MRI,Pelvis,MRI PELVIS,45-49,male,FINDINGS:\nBOWEL: The visualized loops of smal...,The examination is suboptimal due to multiple ...,FINDINGS:\nThe examination is suboptimal due t...


In [13]:
train['template_content'][0].split("\n")

['FINDINGS:',
 'BONES: No acute fracture or focal osseous lesion.',
 'JOINTS: No dislocation. The joint spaces are normal.',
 'SOFT TISSUES: The soft tissues are unremarkable.',
 '',
 'IMPRESSION:',
 'No acute osseous abnormality.']

In [14]:
train['dictation'][0].split("\n")

['No acute fracture or dislocation. Mild right hip osteoarthrosis with mild superolateral joint space narrowing and small marginal acetabular and femoral head osteophytes. Mild degenerative changes of the bilateral sacroiliac joints Mild degenerative changes of the visualized lower lumbar spine. Pelvic ring is intact.']

OLLAMA CLIENT LLM AND EXECUTION

In [35]:
from ollama import Client

client = Client(host="http://localhost:11434")

def get_prompt(inputs):
    with open("prompt.yaml", "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)

    prompt_file = config["prompt"]["selected"]

    with open(f'./prompts/{prompt_file}', "r", encoding="utf-8") as f:
        prompt_text = f.read()

    prompt = prompt_text.format(
        modality=inputs["modality"],
        body_part=inputs["body_part"],
        study_description=inputs["study_description"],
        patient_age_band=inputs["patient_age_band"],
        patient_sex=inputs["patient_sex"],
        template_content=inputs["template_content"],
        dictation=inputs["dictation"]
    )
    
    return prompt

prompt = get_prompt({
    "modality": test['modality'][0],
    "body_part": test['body_part'][0],
    "study_description": test['study_description'][0],
    "patient_age_band": test['patient_age_band'][0],
    "patient_sex": test['patient_sex'][0],
    "template_content": test['template_content'][0],
    "dictation": test['dictation'][0]
})

response = client.generate(
        model="deepseek-r1:1.5b",
        prompt=prompt,
        stream=False
    )

response["response"]

'FINDINGS:\n- GALLBLADDER AND BILIARY TREE: No focal hepatic lesion.\n- SPLEEN: No focal benign lesion.\n- SPLEEN: No focal benign lesion.\n- ABdomINAL WALL: No focal mesenteric or retroperitoneal lymphadenopathy.\n- DESCIANHO: No focal benign lesion.\n- LYMPH NODES: No significant mesenteric or retroperitoneal lymphadenopathy.\n\nIMPRESSION:\n- Colonic diverticulitis present. No extraluminal free air.\n- SPLEEN: No focal benign lesion.\n- No significant mesenteric or retroperitoneal lymphadenopathy.'

In [ ]:
from ollama import Client

client = Client(host="http://localhost:11434")

def callLLM(inputs):

    prompt = get_prompt(inputs)
    
    response = client.generate(
        model="llama3.1:8b",
        prompt=prompt,
        stream=True
    )

    return response["response"]

Here is a concise radiology impression:

"Right basilar opacity, small right pleural effusion."

Or, if you'd like to make it even more concise:

"Mild right basilar opacity and small pleural effusion."


In [22]:
for i in range(10):
    row = test.iloc[i]
    print(row)

case_id                           00320399-b3ed-447c-be07-a3e3b9af6e2e
modality                                                            CT
body_part                                                      Abdomen
study_description                                          CT A/P WO-B
patient_age_band                                                 60-64
patient_sex                                                     female
template_content     FINDINGS:\nLIVER: Normal in size and attenuati...
dictation            Surgically absent gallbladder. Surgical clips ...
Name: 0, dtype: str
case_id                           01b1c9ab-800b-4963-b11d-9d503952e57c
modality                                                          XRAY
body_part                                                 Lumbar spine
study_description                                            XR LSP 4V
patient_age_band                                                 40-44
patient_sex                                              

In [ ]:
for i in range(len(test)):
    row = callLLM(test['template_content'][i], test['dictation'][i])
    print(row)